# TechAdmin Database Validation and Analytics
Read-only notebook for connection checks, schema inventory, pandas summaries, data-quality checks, operation-policy checks, and readiness reporting.

## 1. Setup and shared connection

In [ ]:
from pathlib import Path
import sys
from typing import Any
import pandas as pd
from IPython.display import display
from sqlalchemy import text

here = Path.cwd().resolve()
project_root = next((p for p in [here, *here.parents] if (p / "App").is_dir()), None)
if project_root is None:
    raise RuntimeError("Start Jupyter from inside the TechAdmin repository")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from App.db.connection import DB_SCHEMA, engine
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)
print(f"Project root: {project_root}")
print(f"Configured schema: {DB_SCHEMA}")

## 2. Read-only query helper

In [ ]:
def query_df(sql: str, params: dict[str, Any] | None = None) -> pd.DataFrame:
    # Execute a parameterized SELECT statement and return a DataFrame.
    with engine.connect() as connection:
        return pd.read_sql_query(text(sql), connection, params=params or {})

## 3. Connection health

In [ ]:
connection_df = query_df("""
SELECT current_database() AS database_name,
       current_user AS database_user,
       current_schema() AS current_schema,
       current_setting('search_path') AS search_path,
       inet_server_addr()::text AS server_address,
       inet_server_port() AS server_port
""")
display(connection_df)

## 4. Schema, table, and row-count inventory

In [ ]:
schemas_df = query_df("""
SELECT schema_name FROM information_schema.schemata
WHERE schema_name NOT LIKE 'pg_%' AND schema_name <> 'information_schema'
ORDER BY schema_name
""")
tables_df = query_df("""
SELECT table_schema, table_name, table_type
FROM information_schema.tables
WHERE table_schema = :schema_name
ORDER BY table_name
""", {"schema_name": DB_SCHEMA})
available_tables = set(tables_df["table_name"].tolist())

display(schemas_df)
display(tables_df)

row_counts = []
for table_name in sorted(available_tables):
    result = query_df(f'SELECT COUNT(*) AS row_count FROM "{DB_SCHEMA}"."{table_name}"')
    row_counts.append({"table_name": table_name, "row_count": int(result.loc[0, "row_count"])})
row_counts_df = pd.DataFrame(row_counts)
display(row_counts_df)

## 5. Column inventory
Change `TABLE_NAME` to inspect another discovered table.

In [ ]:
TABLE_NAME = "app_users"
if TABLE_NAME not in available_tables:
    raise ValueError(f"Unknown table: {TABLE_NAME}")
columns_df = query_df("""
SELECT ordinal_position, column_name, data_type, is_nullable, column_default
FROM information_schema.columns
WHERE table_schema = :schema_name AND table_name = :table_name
ORDER BY ordinal_position
""", {"schema_name": DB_SCHEMA, "table_name": TABLE_NAME})
display(columns_df)

## 6. Application users and summary

In [ ]:
app_users_df = query_df(f"""
SELECT user_id, entra_object_id, user_principal_name, display_name,
       department, is_active, created_at, updated_at
FROM "{DB_SCHEMA}"."app_users"
ORDER BY display_name
""")
display(app_users_df)

user_summary_df = pd.DataFrame({
    "metric": ["Registered users", "Active users", "Inactive users", "Departments represented"],
    "value": [
        len(app_users_df),
        int(app_users_df["is_active"].fillna(False).sum()),
        int((~app_users_df["is_active"].fillna(False)).sum()),
        int(app_users_df["department"].dropna().nunique()),
    ],
})
display(user_summary_df)

department_summary_df = (
    app_users_df.assign(department=app_users_df["department"].fillna("UNSPECIFIED"))
    .groupby("department", as_index=False)
    .agg(total_users=("user_id", "count"), active_users=("is_active", "sum"))
)
display(department_summary_df)

## 7. User data-quality checks

In [ ]:
required_users = ["user_id", "entra_object_id", "user_principal_name", "display_name", "is_active", "created_at", "updated_at"]
user_quality_df = pd.DataFrame({
    "check": ["Missing required values", "Duplicate user_id", "Duplicate Entra ID", "Duplicate UPN ignoring case", "Blank UPN", "Blank display name", "Invalid timestamp order"],
    "issue_count": [
        int(app_users_df[required_users].isna().sum().sum()),
        int(app_users_df["user_id"].duplicated().sum()),
        int(app_users_df["entra_object_id"].duplicated().sum()),
        int(app_users_df["user_principal_name"].astype(str).str.strip().str.lower().duplicated().sum()),
        int(app_users_df["user_principal_name"].fillna("").str.strip().eq("").sum()),
        int(app_users_df["display_name"].fillna("").str.strip().eq("").sum()),
        int((pd.to_datetime(app_users_df["updated_at"], utc=True) < pd.to_datetime(app_users_df["created_at"], utc=True)).sum()),
    ],
})
user_quality_df["status"] = user_quality_df["issue_count"].map(lambda n: "PASS" if n == 0 else "REVIEW")
display(user_quality_df)

## 8. Operations catalog and summary

In [ ]:
operations_df = query_df(f"""
SELECT operation_id, operation_code, operation_name, execution_type,
       tool_name, script_name, risk_level, requires_approval, is_active, created_at
FROM "{DB_SCHEMA}"."operations"
ORDER BY operation_code
""")
display(operations_df)

operation_summary_df = pd.DataFrame({
    "metric": ["Registered operations", "Active operations", "Inactive operations", "Operations requiring approval"],
    "value": [
        len(operations_df),
        int(operations_df["is_active"].fillna(False).sum()),
        int((~operations_df["is_active"].fillna(False)).sum()),
        int(operations_df["requires_approval"].fillna(False).sum()),
    ],
})
display(operation_summary_df)
display(operations_df.groupby("execution_type", as_index=False).agg(total_operations=("operation_id", "count")))
display(operations_df.groupby("risk_level", as_index=False).agg(total_operations=("operation_id", "count"), approval_required=("requires_approval", "sum")))

## 9. Operation configuration and suggested policy checks
The final rule is suggested project guidance until formally approved.

In [ ]:
allowed_types = {"API", "SCRIPT", "JOB"}
allowed_risks = {"LOW", "MEDIUM", "HIGH", "CRITICAL"}
required_ops = ["operation_id", "operation_code", "operation_name", "execution_type", "tool_name", "risk_level", "requires_approval", "is_active", "created_at"]
operation_quality_df = pd.DataFrame({
    "check": ["Missing required values", "Duplicate operation_id", "Duplicate operation_code", "Invalid execution type", "Invalid risk level", "SCRIPT missing script_name", "API containing script_name", "Suggested policy: active HIGH/CRITICAL without approval"],
    "issue_count": [
        int(operations_df[required_ops].isna().sum().sum()),
        int(operations_df["operation_id"].duplicated().sum()),
        int(operations_df["operation_code"].str.strip().str.upper().duplicated().sum()),
        int((~operations_df["execution_type"].isin(allowed_types)).sum()),
        int((~operations_df["risk_level"].isin(allowed_risks)).sum()),
        int((operations_df["execution_type"].eq("SCRIPT") & operations_df["script_name"].fillna("").str.strip().eq("")).sum()),
        int((operations_df["execution_type"].eq("API") & operations_df["script_name"].fillna("").str.strip().ne("")).sum()),
        int((operations_df["is_active"].eq(True) & operations_df["risk_level"].isin(["HIGH", "CRITICAL"]) & operations_df["requires_approval"].eq(False)).sum()),
    ],
})
operation_quality_df["status"] = operation_quality_df["issue_count"].map(lambda n: "PASS" if n == 0 else "REVIEW")
display(operation_quality_df)

## 10. Useful operation filters

In [ ]:
print("Active operations")
display(operations_df[operations_df["is_active"].eq(True)])
print("High or critical risk operations")
display(operations_df[operations_df["risk_level"].isin(["HIGH", "CRITICAL"])])
print("Operations requiring approval")
display(operations_df[operations_df["requires_approval"].eq(True)])

## 11. TechAdmin readiness summary
Use this as the manager-friendly overview.

In [ ]:
connection_ok = not connection_df.empty and connection_df.loc[0, "database_name"] == "techadmin_dev" and connection_df.loc[0, "current_schema"] == DB_SCHEMA
schema_ok = DB_SCHEMA in schemas_df["schema_name"].tolist()
readiness_df = pd.DataFrame({
    "area": ["Database connection", "Configured schema", "app_users table", "operations table", "User data quality", "Operation configuration"],
    "status": [
        "PASS" if connection_ok else "REVIEW",
        "PASS" if schema_ok else "REVIEW",
        "PASS" if "app_users" in available_tables else "REVIEW",
        "PASS" if "operations" in available_tables else "REVIEW",
        "PASS" if user_quality_df["issue_count"].sum() == 0 else "REVIEW",
        "PASS" if operation_quality_df["issue_count"].sum() == 0 else "REVIEW",
    ],
    "detail": [
        f"Database={connection_df.loc[0, 'database_name']}", f"Schema={DB_SCHEMA}",
        f"Rows={len(app_users_df)}", f"Rows={len(operations_df)}",
        f"Issues={int(user_quality_df['issue_count'].sum())}",
        f"Issues={int(operation_quality_df['issue_count'].sum())}",
    ],
})
display(readiness_df)

## 12. Safe query explorer and exact-UPN search

In [ ]:
def read_table(table_name: str, limit: int = 100) -> pd.DataFrame:
    # Read only a discovered table and restrict output size.
    if table_name not in available_tables:
        raise ValueError(f"Available tables: {sorted(available_tables)}")
    if not isinstance(limit, int) or not 1 <= limit <= 1000:
        raise ValueError("limit must be between 1 and 1000")
    return query_df(f'SELECT * FROM "{DB_SCHEMA}"."{table_name}" LIMIT {limit}')

display(read_table("operations", 20))

SEARCH_UPN = "aman.3.mishra@coforge.com"
user_search_df = query_df(f"""
SELECT user_id, entra_object_id, user_principal_name, display_name,
       department, is_active, created_at, updated_at
FROM "{DB_SCHEMA}"."app_users"
WHERE LOWER(user_principal_name) = LOWER(:upn)
""", {"upn": SEARCH_UPN.strip()})
display(user_search_df)

## 13. Future analytics readiness

In [ ]:
future_capabilities = {
    "operation_requests": "Request lifecycle and source-channel analytics",
    "operation_executions": "Success, failure, duration, error, and retry analytics",
    "approval_requests": "Approval status, expiry, and turnaround analytics",
}
future_readiness_df = pd.DataFrame([
    {"table_name": name, "available": name in available_tables, "future_capability": capability}
    for name, capability in future_capabilities.items()
])
display(future_readiness_df)

# How to use and leverage the notebook

## During development
1. Run all cells after model, seed, or migration changes.
2. Confirm the readiness summary shows `PASS`.
3. Investigate quality checks marked `REVIEW`.
4. Use the table explorer and UPN search for troubleshooting.
5. Validate active, high-risk, and approval-controlled operations.

## Before a demo or update
1. Run the connection, inventory, summary, quality, and readiness sections.
2. Present the readiness table as the top-level status.
3. Use calculated summaries instead of manually inspecting PostgreSQL.
4. Clear outputs before sharing.

## Before and after migrations
1. Run the notebook before migration.
2. Apply the migration.
3. Rerun table inventory, row counts, and quality checks.
4. Investigate unexpected changes.

## Safety and Git hygiene
- Keep the notebook read-only.
- Never add passwords, tokens, or secrets.
- Do not commit outputs containing UPNs, Entra IDs, or server details.
- Before committing, use **Restart Kernel and Clear All Outputs**, then save.
- Keep `.env` ignored.